# Notebook 04 — CNN Training

**Project:** Wafer Defect Detection & Yield Risk Dashboard  
**Phase:** Phase 4 — CNN Model Training

## Goals

1. Build a PyTorch Dataset from preprocessed wafer maps
2. Train a custom CNN (WaferCNN) with class-weighted loss
3. Monitor training/validation metrics per epoch
4. Save the best checkpoint by val Macro F1
5. Plot training curves
6. Evaluate best model on the held-out test set
7. Compare against baseline (Macro F1 > 0.5515)

## Architecture Summary

```
Input: (B, 1, 64, 64)
  -> Conv(32) + BN + ReLU + MaxPool  ->  (B, 32, 32, 32)
  -> Conv(64) + BN + ReLU + MaxPool  ->  (B, 64, 16, 16)
  -> Conv(128) + BN + ReLU + MaxPool ->  (B, 128, 8, 8)
  -> Flatten  ->  (B, 8192)
  -> Linear(256) + ReLU + Dropout(0.5)
  -> Linear(9)   [logits]
```
Total parameters: 2,192,841

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.dataset import make_dataloaders, get_class_weights_tensor
from src.model import WaferCNN, count_parameters
from src.train import train, evaluate_epoch
from src.evaluate import compute_metrics, print_metrics, plot_confusion_matrix
from src.data_loader import DEFECT_CLASSES
from torch.nn import CrossEntropyLoss

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

MODELS_DIR = Path('../models')
OUTPUT_DIR = Path('../outputs/figures')

## 1. Build DataLoaders

In [ ]:
train_loader, val_loader, test_loader = make_dataloaders(
    processed_dir='../data/processed',
    batch_size=256,
    num_workers=0,
    augment_train=True,
)
print(f'Train batches: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}')

## 2. Initialise Model

In [ ]:
model = WaferCNN(num_classes=9, dropout=0.5).to(DEVICE)
print(f'Parameters: {count_parameters(model):,}')
print(model)

## 3. Train

Training is saved to `models/best_model.pth` (best val Macro F1).  
Early stopping triggers after 7 epochs without improvement.

In [ ]:
class_weights = get_class_weights_tensor('../data/processed', device=DEVICE)

history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    class_weights=class_weights,
    num_epochs=30,
    lr=1e-3,
    patience=7,
    save_path=MODELS_DIR / 'best_model.pth',
)

## 4. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   color='orange')
axes[0].set_title('Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['val_f1'],  label='Val Macro F1',  color='green')
axes[1].plot(epochs, history['val_acc'], label='Val Accuracy',   color='purple')
axes[1].axhline(0.5515, color='red', linestyle='--', alpha=0.7, label='Baseline F1 (LR)')
axes[1].set_title('Validation Metrics', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('WaferCNN Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cnn_training_curves.png', dpi=150)
plt.show()

## 5. Evaluate Best Model on Test Set

In [ ]:
model.load_state_dict(torch.load(MODELS_DIR / 'best_model.pth', map_location=DEVICE))
criterion = CrossEntropyLoss(weight=class_weights)

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in test_loader:
        logits = model(X.to(DEVICE))
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(y.numpy())

metrics_cnn = compute_metrics(np.array(all_labels), np.array(all_preds))
print_metrics(metrics_cnn, 'WaferCNN -- Test Set')

In [ ]:
plot_confusion_matrix(
    np.array(all_labels), np.array(all_preds),
    title='WaferCNN -- Confusion Matrix (Test Set, Normalized)',
    save_path=OUTPUT_DIR / 'confusion_matrix_cnn.png',
)

## 6. Comparison vs Baseline

In [ ]:
print(f"{'Model':<28} {'Accuracy':>10} {'Macro F1':>10}")
print('-' * 52)
print(f"{'SGD LogReg (baseline)':<28} {'0.8803':>10} {'0.5515':>10}")
print(f"{'WaferCNN':<28} {metrics_cnn['accuracy']:>10.4f} {metrics_cnn['macro_f1']:>10.4f}")

print(f"\nPer-class F1 comparison:")
print(f"{'Class':<15} {'Baseline F1':>12} {'CNN F1':>10} {'Support':>10}")
print('-' * 52)
baseline_f1 = {'Center':0.8232,'Donut':0.4558,'Edge-Loc':0.4321,'Edge-Ring':0.9020,
               'Loc':0.2275,'Near-full':0.4286,'Random':0.6641,'Scratch':0.0880,'none':0.9420}
for cls in DEFECT_CLASSES:
    b = baseline_f1[cls]
    c = metrics_cnn['per_class'][cls]['f1']
    s = metrics_cnn['per_class'][cls]['support']
    arrow = ' (+)' if c > b else ' (-)'
    print(f"{cls:<15} {b:>12.4f} {c:>10.4f} {s:>10,}{arrow}")